# Colorisation du nuage de points Lidar depuis l'orthophotographie

Ce notebook vérifie que `processing.lidar.colorize.colorize_from_raster` (PDAL `filters.colorization`) assigne correctement les dimensions Red/Green/Blue à chaque point à partir de l'orthophotographie IGN, sur le nuage fusionné produit par le notebook 15.

Une orthophoto réelle (`orthophoto_lidar_tile.tif`) a été téléchargée spécifiquement sur l'emprise de la dalle Lidar disponible (886000–887000 / 6499000–6500000), afin de disposer d'une couverture réelle complète pour ce test.

Comme convenu, le fichier traité (colorisé) est écrit dans `data/processed/raster/lidar/`, et non à côté de la source brute.

In [ ]:
import json
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pdal
import rasterio

from processing.lidar.colorize import colorize_from_raster

merged_path = repo_root / "data" / "raw" / "raster" / "lidar" / "nuage_points.las"
raster_path = repo_root / "data" / "raw" / "raster" / "orthophoto" / "orthophoto_lidar_tile.tif"
output_path = repo_root / "data" / "processed" / "raster" / "lidar" / "nuage_points_colorise.las"

## 1. Colorisation du nuage de points

In [ ]:
result = colorize_from_raster(merged_path, raster_path, output_path=output_path)
print("Fichier colorise :", result)
print(f"Taille : {result.stat().st_size / 1_000_000:.1f} Mo")

## 2. Vérification : dimensions RGB et cohérence avec l'orthophoto

In [ ]:
pipeline = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(result)}]))
count = pipeline.execute()
arr = pipeline.arrays[0]

print("Points colorises :", count)
print("Dimensions :", arr.dtype.names)
print("Red   :", arr["Red"].min(), "-", arr["Red"].max())
print("Green :", arr["Green"].min(), "-", arr["Green"].max())
print("Blue  :", arr["Blue"].min(), "-", arr["Blue"].max())

# Verification ponctuelle sur plusieurs points : la couleur du point doit
# correspondre exactement au pixel de l'orthophoto sous ses coordonnees X, Y.
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(arr), size=5, replace=False)

with rasterio.open(raster_path) as src:
    print("\nComparaison point vs pixel raster :")
    for idx in sample_idx:
        x, y = float(arr["X"][idx]), float(arr["Y"][idx])
        row, col = src.index(x, y)
        pixel = src.read(window=((row, row + 1), (col, col + 1)))
        point_rgb = (arr["Red"][idx], arr["Green"][idx], arr["Blue"][idx])
        pixel_rgb = (pixel[0, 0, 0], pixel[1, 0, 0], pixel[2, 0, 0])
        match = "OK" if tuple(point_rgb) == tuple(pixel_rgb) else "DIFFERENT"
        print(f"  point {idx} : couleur={point_rgb}, pixel={pixel_rgb} -> {match}")

## 3. Aperçu du nuage colorisé (couleurs réelles)

In [ ]:
import matplotlib.pyplot as plt

sample_plot_idx = rng.choice(len(arr), size=min(300_000, len(arr)), replace=False)
sample = arr[sample_plot_idx]
colors = np.stack([sample["Red"], sample["Green"], sample["Blue"]], axis=1) / 255.0

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(sample["X"], sample["Y"], c=colors, s=0.5)
ax.set_aspect("equal")
ax.set_title("Nuage de points colorise depuis l'orthophoto (couleurs reelles)")
plt.show()